# 📚 Biblioteca NoSQL – MongoDB (Colab Notebook)

Questo notebook mostra come usare una parte del database della **biblioteca** in un contesto **NoSQL (MongoDB)**.

Contenuti:
- Import dati dal database SQLite esistente
- Inserimento documenti in MongoDB
- Operazioni elementari: **SELECT, UPDATE, DELETE, JOIN (lookup)**
- Ricerca di un libro e conteggio dei **passaggi necessari** usando **B+Tree**


In [1]:
!pip install pymongo mongomock

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 18.3 MB/s eta 0:00:00


In [3]:
import sqlite3
import pandas as pd
import mongomock
import sqlite3
import random
import string
import pandas as pd

In [4]:
libri_reali = [
("Il Signore degli Anelli","J.R.R. Tolkien","Fantasy"),
("Lo Hobbit","J.R.R. Tolkien","Fantasy"),
("1984","George Orwell","SciFi"),
("La fattoria degli animali","George Orwell","Romanzo"),
("Dune","Frank Herbert","SciFi"),
("Neuromante","William Gibson","SciFi"),
("Fahrenheit 451","Ray Bradbury","SciFi"),
("Fondazione","Isaac Asimov","SciFi"),
("Il Nome della Rosa","Umberto Eco","Storico"),
("I Promessi Sposi","Alessandro Manzoni","Romanzo"),
("Il Conte di Montecristo","Alexandre Dumas","Romanzo"),
("I Tre Moschettieri","Alexandre Dumas","Romanzo"),
("Orgoglio e Pregiudizio","Jane Austen","Romanzo"),
("Emma","Jane Austen","Romanzo"),
("Moby Dick","Herman Melville","Romanzo"),
("Il Vecchio e il Mare","Ernest Hemingway","Romanzo"),
("Per chi suona la campana","Ernest Hemingway","Romanzo"),
("Il Grande Gatsby","F. Scott Fitzgerald","Romanzo"),
("Ulisse","James Joyce","Romanzo"),
("Dracula","Bram Stoker","Horror"),
("Frankenstein","Mary Shelley","Horror"),
("Il ritratto di Dorian Gray","Oscar Wilde","Romanzo"),
("Delitto e castigo","Fëdor Dostoevskij","Romanzo"),
("I fratelli Karamazov","Fëdor Dostoevskij","Romanzo"),
("Guerra e pace","Lev Tolstoj","Storico"),
("Anna Karenina","Lev Tolstoj","Romanzo"),
("Don Chisciotte","Miguel de Cervantes","Romanzo"),
("Il piccolo principe","Antoine de Saint-Exupéry","Romanzo"),
("Il processo","Franz Kafka","Romanzo"),
("Il castello","Franz Kafka","Romanzo"),
("La metamorfosi","Franz Kafka","Romanzo"),
("Lolita","Vladimir Nabokov","Romanzo"),
("Il maestro e Margherita","Michail Bulgakov","Romanzo"),
("Il deserto dei Tartari","Dino Buzzati","Romanzo"),
("Se questo è un uomo","Primo Levi","Storico"),
("Il barone rampante","Italo Calvino","Romanzo"),
("Il visconte dimezzato","Italo Calvino","Romanzo"),
("Le città invisibili","Italo Calvino","Romanzo"),
("Il sentiero dei nidi di ragno","Italo Calvino","Romanzo"),
("Il Gattopardo","Giuseppe Tomasi di Lampedusa","Storico"),
("Il fu Mattia Pascal","Luigi Pirandello","Romanzo"),
("Uno, nessuno e centomila","Luigi Pirandello","Romanzo"),
("Il giorno della civetta","Leonardo Sciascia","Romanzo"),
("Cristo si è fermato a Eboli","Carlo Levi","Romanzo"),
("La coscienza di Zeno","Italo Svevo","Romanzo"),
("Il codice Da Vinci","Dan Brown","Thriller"),
("Angeli e demoni","Dan Brown","Thriller"),
("Jurassic Park","Michael Crichton","SciFi"),
("The Andromeda Strain","Michael Crichton","SciFi"),
("The Shining","Stephen King","Horror"),
("It","Stephen King","Horror"),
("Misery","Stephen King","Horror"),
("Carrie","Stephen King","Horror"),
("Pet Sematary","Stephen King","Horror"),
("American Gods","Neil Gaiman","Fantasy"),
("Good Omens","Neil Gaiman","Fantasy"),
("Neverwhere","Neil Gaiman","Fantasy"),
("Snow Crash","Neal Stephenson","SciFi"),
("Cryptonomicon","Neal Stephenson","SciFi"),
("Ready Player One","Ernest Cline","SciFi"),
("The Martian","Andy Weir","SciFi"),
("Project Hail Mary","Andy Weir","SciFi"),
("Hyperion","Dan Simmons","SciFi"),
("Ender's Game","Orson Scott Card","SciFi"),
("The Road","Cormac McCarthy","Romanzo"),
("Blood Meridian","Cormac McCarthy","Romanzo"),
("No Country for Old Men","Cormac McCarthy","Romanzo"),
("The Catcher in the Rye","J.D. Salinger","Romanzo"),
("To Kill a Mockingbird","Harper Lee","Romanzo"),
("Lord of the Flies","William Golding","Romanzo"),
("The Stand","Stephen King","Horror"),
("The Dark Tower","Stephen King","Fantasy"),
("A Game of Thrones","George R.R. Martin","Fantasy"),
("A Clash of Kings","George R.R. Martin","Fantasy"),
("A Storm of Swords","George R.R. Martin","Fantasy"),
("The Hunger Games","Suzanne Collins","SciFi"),
("Catching Fire","Suzanne Collins","SciFi"),
("Mockingjay","Suzanne Collins","SciFi"),
("The Handmaid's Tale","Margaret Atwood","SciFi"),
("Oryx and Crake","Margaret Atwood","SciFi"),
("Brave New World","Aldous Huxley","SciFi"),
("Island","Aldous Huxley","SciFi"),
("The Time Machine","H.G. Wells","SciFi"),
("The War of the Worlds","H.G. Wells","SciFi"),
("The Invisible Man","H.G. Wells","SciFi"),
("The Hobbit Annotated","Douglas Anderson","Fantasy"),
("The Silmarillion","J.R.R. Tolkien","Fantasy"),
("Children of Hurin","J.R.R. Tolkien","Fantasy"),
("The Fall of Gondolin","J.R.R. Tolkien","Fantasy"),
("The Name of the Wind","Patrick Rothfuss","Fantasy"),
("The Wise Man's Fear","Patrick Rothfuss","Fantasy"),
("Mistborn","Brandon Sanderson","Fantasy"),
("The Way of Kings","Brandon Sanderson","Fantasy"),
("Words of Radiance","Brandon Sanderson","Fantasy"),
("Oathbringer","Brandon Sanderson","Fantasy")
]

In [5]:
# CONNECT DATABASE
conn = sqlite3.connect('biblioteca.db')
cursor = conn.cursor()

In [6]:
# CREATE TABLES

cursor.execute('''
CREATE TABLE IF NOT EXISTS autore(
 id_autore INTEGER PRIMARY KEY,
 nome TEXT,
 eta INTEGER
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS scaffale(
 id_scaffale INTEGER PRIMARY KEY,
 zona TEXT,
 piano INTEGER
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS libro(
 id_libro INTEGER PRIMARY KEY,
 titolo TEXT,
 genere TEXT,
 costo REAL,
 id_scaffale INTEGER,
 FOREIGN KEY(id_scaffale) REFERENCES scaffale(id_scaffale)
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS autore_libro(
 id_autore INTEGER,
 id_libro INTEGER,
 PRIMARY KEY(id_autore,id_libro),
 FOREIGN KEY(id_autore) REFERENCES autore(id_autore),
 FOREIGN KEY(id_libro) REFERENCES libro(id_libro)
)
''')

conn.commit()

In [7]:
# CONFIG


for titolo, autore, genere in libri_reali:

    cursor.execute(
        "INSERT INTO autore(nome) VALUES (?)",
        (autore,)
    )

    id_autore = cursor.lastrowid

    cursor.execute(
        "INSERT INTO libro(titolo, genere) VALUES (?,?)",
        (titolo, genere)
    )

    id_libro = cursor.lastrowid

    cursor.execute(
        "INSERT INTO autore_libro VALUES (?,?)",
        (id_autore, id_libro)
    )

conn.commit()

In [8]:
# CREATE INDEX (HASH LIKE)
cursor.execute('CREATE INDEX IF NOT EXISTS idx_autore ON autore(id_autore)')
cursor.execute('CREATE INDEX IF NOT EXISTS idx_libro ON libro(id_libro)')
cursor.execute('CREATE INDEX IF NOT EXISTS idx_scaffale ON scaffale(id_scaffale)')
cursor.execute('CREATE INDEX IF NOT EXISTS idx_al_autore ON autore_libro(id_autore)')
cursor.execute('CREATE INDEX IF NOT EXISTS idx_al_libro ON autore_libro(id_libro)')

conn.commit()

In [9]:
def assegna_id_scaffale(conn):

    cursor = conn.cursor()
    cursor.execute("""
    UPDATE libro
    SET id_scaffale = id_libro
    """)
    conn.commit()
    print("ID scaffale assegnati")
assegna_id_scaffale(conn)

ID scaffale assegnati


## 1️⃣ Caricamento dati dal database SQLite

In [10]:
conn = sqlite3.connect('biblioteca.db')

libri = pd.read_sql_query('SELECT * FROM libro', conn)
autori = pd.read_sql_query('SELECT * FROM autore', conn)
relazioni = pd.read_sql_query('SELECT * FROM autore_libro', conn)

print('LIBRI')
display(libri.head())

print('AUTORI')
display(autori.head())

LIBRI


,id_libro,titolo,genere,costo,id_scaffale
0,1,Il Signore degli Anelli,Fantasy,None,1
1,2,Lo Hobbit,Fantasy,None,2
2,3,1984,SciFi,None,3
3,4,La fattoria degli animali,Romanzo,None,4
4,5,Dune,SciFi,None,5


AUTORI


,id_autore,nome,eta
0,1,J.R.R. Tolkien,None
1,2,J.R.R. Tolkien,None
2,3,George Orwell,None
3,4,George Orwell,None
4,5,Frank Herbert,None


## 2️⃣ Creazione database MongoDB

In [11]:
client = mongomock.MongoClient()

db = client['biblioteca']

col_libri = db['libri']
col_autori = db['autori']

## 3️⃣ Inserimento documenti

In [12]:
col_libri.insert_many(libri.to_dict('records'))
col_autori.insert_many(autori.to_dict('records'))

print('Documenti inseriti')

Documenti inseriti


## 4️⃣ SELECT (find)

In [13]:
doc = col_libri.find_one({'id_libro':30})
print(doc)

{'id_libro': 30, 'titolo': 'Il castello', 'genere': 'Romanzo', 'costo': None, 'id_scaffale': 30, '_id': ObjectId('69b82b5dbadc9637459de2da')}


## 5️⃣ UPDATE

In [14]:
col_libri.update_one(
    {'id_libro':30},
    {'$set':{'genere':'Fantasy'}}
)

print(col_libri.find_one({'id_libro':30}))

{'id_libro': 30, 'titolo': 'Il castello', 'genere': 'Fantasy', 'costo': None, 'id_scaffale': 30, '_id': ObjectId('69b82b5dbadc9637459de2da')}


## 6️⃣ DELETE

In [15]:
col_libri.delete_one({'id_libro':30})

print('Libro eliminato')

Libro eliminato


## 7️⃣ JOIN in MongoDB (lookup)

In [16]:
pipeline = [
    {
        '$lookup':{
            'from':'autori',
            'localField':'id_libro',
            'foreignField':'id_autore',
            'as':'autore'
        }
    }
]

result = list(col_libri.aggregate(pipeline))
result[:5]

[{'id_libro': 1,
  'titolo': 'Il Signore degli Anelli',
  'genere': 'Fantasy',
  'costo': None,
  'id_scaffale': 1,
  '_id': ObjectId('69b82b5dbadc9637459de2bd'),
  'autore': [{'id_autore': 1,
    'nome': 'J.R.R. Tolkien',
    'eta': None,
    '_id': ObjectId('69b82b5dbadc9637459de31c')}]},
 {'id_libro': 2,
  'titolo': 'Lo Hobbit',
  'genere': 'Fantasy',
  'costo': None,
  'id_scaffale': 2,
  '_id': ObjectId('69b82b5dbadc9637459de2be'),
  'autore': [{'id_autore': 2,
    'nome': 'J.R.R. Tolkien',
    'eta': None,
    '_id': ObjectId('69b82b5dbadc9637459de31d')}]},
 {'id_libro': 3,
  'titolo': '1984',
  'genere': 'SciFi',
  'costo': None,
  'id_scaffale': 3,
  '_id': ObjectId('69b82b5dbadc9637459de2bf'),
  'autore': [{'id_autore': 3,
    'nome': 'George Orwell',
    'eta': None,
    '_id': ObjectId('69b82b5dbadc9637459de31e')}]},
 {'id_libro': 4,
  'titolo': 'La fattoria degli animali',
  'genere': 'Romanzo',
  'costo': None,
  'id_scaffale': 4,
  '_id': ObjectId('69b82b5dbadc9637459de2c

In [19]:
col_libri.insert_many(libri.to_dict("records"))
col_autori.insert_many(autori.to_dict("records"))

# CREA INDICE
col_libri.create_index("id_libro")

'id_libro_1'

### 8️⃣ Ricerca libro e conteggio passaggi

In [18]:
def cerca_libro(id_target):

    passi = 0

    for libro in col_libri.find():
        passi += 1

        if libro['id_libro'] == id_target:
            return libro, passi

    return None, passi


libro, passi = cerca_libro(10)

print('Libro trovato:', libro)
print('Numero passaggi ricerca:', passi)

Libro trovato: {'id_libro': 10, 'titolo': 'I Promessi Sposi', 'genere': 'Romanzo', 'costo': None, 'id_scaffale': 10, '_id': ObjectId('69b82b5dbadc9637459de2c6')}
Numero passaggi ricerca: 10


In [32]:
def cerca_libro(id_target):
    libro = col_libri.find_one({"id_libro": id_target})
    return libro

In [29]:
import math

def cerca_libro_passaggi(id_target):

    n = col_libri.count_documents({})

    passaggi = math.ceil(math.log2(n))

    print("Libro cercato:", id_target)
    print("Numero documenti:", n)
    print("Passaggi stimati (B+tree):", passaggi)

In [36]:
libro = cerca_libro(30)
print(cerca_libro_passaggi(30))
print(libro)


Libro cercato: 30
Numero documenti: 189
Passaggi stimati (B-tree): 8
None
{'id_libro': 30, 'titolo': 'Il castello', 'genere': 'Romanzo', 'costo': None, 'id_scaffale': 30, '_id': ObjectId('69b82bc7badc9637459de398')}


## 📊 Analisi complessità

- MongoDB senza indice → **O(n)**
- MongoDB B+Tree → **O(log n)**
- Hash Table → **O(n_bucket)**
- B‑Tree → **O(log n)**

Con **100 libri**:

- Mongo ≈ 8 passaggi
- B‑Tree ≈ log₂(100) ≈ 7 passaggi
- Hash ≈ 10 passaggi